# EmbedMed

# ------ 

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from controllers.ProcessController import ProcessController

config = {
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY"),
    "GENERATION_MODEL": "openai/gpt-oss-120b",
    "EMBEDDING_MODEL": "BAAI/bge-small-en-v1.5",
    "VECTOR_DB_PATH": "chroma_db",
}

controller = ProcessController(config)

c:\Users\ALNOUR\anaconda3\envs\sic\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### loading and chunking

In [2]:
controller.load_pdf(
    pdf_path="assets/hypertension-in-adults-diagnosis-and-management.pdf",
    document_id="NICE-NG136-2026",
    title="Hypertension in Adults: Diagnosis and Management",
    version="NG136",
    publication_date="2019-08-28"
)

controller.chunk_documents(document_id="NICE-NG136-2026")

controller.build_vectorstore(collection_name="hypertension_clinical_kb")

print("Pipeline completed successfully")
print(f"Pages loaded: {len(controller.pages)}")
print(f"Chunks created: {len(controller.chunks)}")

Pipeline completed successfully
Pages loaded: 52
Chunks created: 156


### testing

In [3]:
result = controller.ask("What is the target blood pressure for people with type 2 diabetes?")
print(result["answer"])
print("\nSources:")
for s in result["retrieved_sources"]:
    print(s)

The guideline recommends a clinic blood‑pressure target of **below 140/90 mmHg** for people with type 2 diabetes (as for other adults with hypertension) [Document ID | p. 14 | NICE‑NG136‑2026‑CH‑038]. The committee noted that there is no evidence to support a different, lower target for type 2 diabetes, and that the previous lower target (<130/80 mmHg) was based on limited data [Document ID | p. 38 | NICE‑NG136‑2026‑CH‑113].  

Educational information only; not a diagnosis or medical advice.

Sources:
{'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113', 'preview': 'on people already receiving treatment and that it lacked information on adverse events.  The committee agreed that there was no evidence to suggest that blood pressure targets  sho'}
{'document_id': 'NICE-NG136-2026', 'page': 14, 'chunk_id': 'NICE-NG136-2026-CH-038', 'preview': 'hypertension in pregnancy.  See also table 1 for clinic blood pressure targets for people aged under 80 and table 2 

### 20 questions evaluation and displaying chnk info

In [9]:
import sys
sys.path.append("evaluation")

from evaluation.eval_questions import EVAL_QUESTIONS
from evaluation.run_evaluation import run_retrieval, compare_k_values

sample_results = run_retrieval(controller, EVAL_QUESTIONS, k=5)

for r in sample_results:
    print("Question:", r["question"])
    for chunk in r["retrieved_chunks"]:
        print(f"  [{chunk['chunk_id']}] score={chunk['score']} page={chunk['page']}")
        print(f"  {chunk['chunk_text'][:100]}...")
    print()

Question: What is the target blood pressure for adults under 80 without diabetes?
  [NICE-NG136-2026-CH-038] score=0.8347 page=14
  hypertension in pregnancy. 
See also table 1 for clinic blood pressure targets for people aged under...
  [NICE-NG136-2026-CH-044] score=0.8283 page=16
  below 140/90 mmHg and ensure that it is maintained below that level. See also 
table 1 for guidance ...
  [NICE-NG136-2026-CH-115] score=0.8245 page=39
  people without hypertension. They also had concerns about the relevance of the study 
design. The co...
  [NICE-NG136-2026-CH-045] score=0.7989 page=16
  hypertension, use the average blood pressure level taken during the person's 
usual waking hours (se...
  [NICE-NG136-2026-CH-119] score=0.7948 page=40
  Blood pressure targets for people with cardiovascular disease 
Recommendation 1.4.23 
Why the commit...

Question: What is the target blood pressure for adults aged 80 and over?
  [NICE-NG136-2026-CH-044] score=0.851 page=16
  below 140/90 mmHg and ens

# comparing 3 diffirent ks

In [7]:
sample_questions = EVAL_QUESTIONS[:3]
k_comparison = compare_k_values(controller, sample_questions)

for question, k_results in k_comparison.items():
    print("=" * 80)
    print("Question:", question)
    for k, chunks in k_results.items():
        print(f"\n  --- k={k} ({len(chunks)} results) ---")
        for c in chunks:
            print(f"  [{c['chunk_id']}] page={c['page']} score={c['score']}")

Question: What is the target blood pressure for adults under 80 without diabetes?

  --- k=3 (3 results) ---
  [NICE-NG136-2026-CH-038] page=14 score=0.8347
  [NICE-NG136-2026-CH-044] page=16 score=0.8283
  [NICE-NG136-2026-CH-115] page=39 score=0.8245

  --- k=5 (5 results) ---
  [NICE-NG136-2026-CH-038] page=14 score=0.8347
  [NICE-NG136-2026-CH-044] page=16 score=0.8283
  [NICE-NG136-2026-CH-115] page=39 score=0.8245
  [NICE-NG136-2026-CH-045] page=16 score=0.7989
  [NICE-NG136-2026-CH-119] page=40 score=0.7948

  --- k=10 (10 results) ---
  [NICE-NG136-2026-CH-038] page=14 score=0.8347
  [NICE-NG136-2026-CH-044] page=16 score=0.8283
  [NICE-NG136-2026-CH-115] page=39 score=0.8245
  [NICE-NG136-2026-CH-045] page=16 score=0.7989
  [NICE-NG136-2026-CH-119] page=40 score=0.7948
  [NICE-NG136-2026-CH-113] page=38 score=0.7912
  [NICE-NG136-2026-CH-146] page=48 score=0.788
  [NICE-NG136-2026-CH-077] page=28 score=0.7871
  [NICE-NG136-2026-CH-155] page=52 score=0.7853
  [NICE-NG136-2026-C

### Compare 2 chunks configs

In [ ]:
controller_small = ProcessController(config)

# 2. نحمل نفس الملف (مع إضافة البارامترز المطلوبة)
controller_small.load_pdf(
    pdf_path="assets/hypertension-in-adults-diagnosis-and-management.pdf",
    document_id="NICE-NG136-2026",
    title="Hypertension in Adults: Diagnosis and Management",
    version="NG136",
    publication_date="2019-08-28"
)

# 3. هنجرب Chunk size أصغر (مثلاً 500)
controller_small.chunk_documents(document_id="NICE-NG136-2026", chunk_size=500, chunk_overlap=50)

# 4. نعمل Collection جديدة في قاعدة البيانات باسم مختلف عشان متدخلش في القديمة
controller_small.build_vectorstore(collection_name="hypertension_small_chunks")

print(f"Original Chunks count: {len(controller.chunks)}")
print(f"Small Chunks count: {len(controller_small.chunks)}")

# 5. نقارن نتيجة نفس السؤال بين الإعدادين
test_q = "What is the target blood pressure for people with type 2 diabetes?"

print("\n=== Results from ORIGINAL chunks ===")
res1 = controller.ask(test_q)
for s in res1["retrieved_sources"][:2]: 
    # حولناها لـ String الأول عشان نقدر نقص منها أول 150 حرف
    print("-", str(s)[:150], "...")

print("\n=== Results from SMALL chunks ===")
res2 = controller_small.ask(test_q)
for s in res2["retrieved_sources"][:2]:
    # حولناها لـ String الأول
    print("-", str(s)[:150], "...")

Original Chunks count: 156
Small Chunks count: 240

=== Results from ORIGINAL chunks ===
- {'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113', 'preview': 'on people already receiving treatment and that it lac ...
- {'document_id': 'NICE-NG136-2026', 'page': 14, 'chunk_id': 'NICE-NG136-2026-CH-038', 'preview': 'hypertension in pregnancy.  See also table 1 for clin ...

=== Results from SMALL chunks ===
- {'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-171', 'preview': 'The committee agreed that there was no evidence to su ...
- {'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-166', 'preview': 'targets should be used. The committee agreed that in  ...


### labeling and calculation of percision3,5 and avg

In [15]:
sample_questions = EVAL_QUESTIONS[:3]
eval_results = run_retrieval(controller, sample_questions, k=5)

for r in eval_results:
    print("=" * 100)
    print(f"Question: {r['question']}")
    print("=" * 100)
    
    for i, chunk in enumerate(r["retrieved_chunks"], 1):
        print(f"\n--- Chunk {i} | ID: [{chunk['chunk_id']}] | Page: {chunk['page']} ---")
        print(chunk['chunk_text'])
        print("-" * 50)

Question: What is the target blood pressure for adults under 80 without diabetes?

--- Chunk 1 | ID: [NICE-NG136-2026-CH-038] | Page: 14 ---
hypertension in pregnancy. 
See also table 1 for clinic blood pressure targets for people aged under 80 and table 2 for 
clinic blood pressure targets for people aged 80 and over. The tables cover people with 
hypertension (with or without type 2 diabetes) as well as people with chronic kidney 
disease or type 1 diabetes. 
Table 1: Clinic blood pressure targets for people aged under 80 
Person under 80 with: 
Clinic blood 
pressure 
target 
Source 
• hypertension (with or without type 2 
diabetes) or 
• type 1 diabetes plus albumin to 
creatinine ratio less than 70 mg/mmol 
or 
• chronic kidney disease plus albumin 
to creatinine ratio less than 70 mg/
mmol 
Below 
140/90 
Recommendation 1.4.20 
NICE's guideline on type 1 
diabetes in adults 
(recommendation 1.13.8) 
NICE's guideline on chronic kidney 
disease (recommendation 1.6.1)
--------------

In [17]:
import pandas as pd

# Task 5: Manually label chunks (1 = Relevant, 0 = Not Relevant)
manual_labels = {
    "Q1: Target BP under 80 without diabetes": [1, 1, 1, 1, 0],
    "Q2: Target BP adults aged 80 and over":   [1, 1, 0, 0, 0],
    "Q3: Target BP people with type 2 diabetes":[0, 1, 0, 0, 1] 
}

# Task 6: Calculate Precision@3 and Precision@5, plus averages
def calc_precision_at_k(labels, k):
    return sum(labels[:k]) / k

results_data = []

for question, labels in manual_labels.items():
    p3 = calc_precision_at_k(labels, 3)
    p5 = calc_precision_at_k(labels, 5)
    results_data.append({
        "Question": question,
        "P@3": round(p3, 2),
        "P@5": round(p5, 2)
    })

eval_df = pd.DataFrame(results_data)

print("=== Evaluation Metrics ===")
print(eval_df.to_string(index=False))

print("\n=== Averages ===")
print(f"Average Precision@3: {eval_df['P@3'].mean():.2f}")
print(f"Average Precision@5: {eval_df['P@5'].mean():.2f}")

=== Evaluation Metrics ===
                                 Question  P@3  P@5
  Q1: Target BP under 80 without diabetes 1.00  0.8
    Q2: Target BP adults aged 80 and over 0.67  0.4
Q3: Target BP people with type 2 diabetes 0.33  0.4

=== Averages ===
Average Precision@3: 0.67
Average Precision@5: 0.53


### Failure Case

#### Task 7: Retrieval Failure Case Analysis

* **Question:** *"What is the target blood pressure for people with type 2 diabetes?"*
* **Failure Mode:** `False Positive Distractor` & `Semantic Drift`
* **Observed Behavior:** The retriever assigned the highest score (Rank 1) to Chunk `[NICE-NG136-2026-CH-113]`. While this chunk perfectly matched the keywords ("type 2 diabetes", "blood pressure targets"), it was actually a "Rationale and Discussion" section explaining the *lack of evidence* for different targets, rather than stating the current actionable target. The actual correct answer was pushed to Rank 2 and Rank 5.
* **Root Cause:** The embedding model successfully captured semantic similarity based on medical terminology but failed to distinguish between the **context of a medical recommendation** versus the **context of a committee discussion/historical evidence**. It couldn't grasp the "intent" of finding a specific numerical target over general discourse.

### keyword search

In [ ]:
# Task 8 (Optional): Try Keyword Search on the failure case question
test_question = "What is the target blood pressure for people with type 2 diabetes?"
keywords = ["target", "type 2 diabetes", "0"] # ضفنا الرقم ككلمة مفتاحية لنرى مدى دقتها
140/9
print(f"=== Keyword Search Results ===\n")
print(f"Question: {test_question}\n")

keyword_results = []

# البحث المباشر بالكلمات في كل الـ Chunks الموجودة في الذاكرة
for chunk in controller.chunks:
    text_lower = chunk.page_content.lower()
    # التأكد أن كل الكلمات المفتاحية موجودة في القطعة
    if "type 2 diabetes" in text_lower and "target" in text_lower:
        keyword_results.append(chunk)

# طباعة أول 3 نتائج
if not keyword_results: 
    print("No chunks found with these exact keywords.")
else:
    for i, chunk in enumerate(keyword_results[:3], 1):
        print(f"--- Keyword Result {i} | ID: [{chunk.metadata.get('chunk_id')}] | Page: {chunk.metadata.get('page_number')} ---")
        print(chunk.page_content[:300].replace('\n', ' '))
        print("-" * 80)

=== Keyword Search Results ===

Question: What is the target blood pressure for people with type 2 diabetes?

--- Keyword Result 1 | ID: [NICE-NG136-2026-CH-038] | Page: 14 ---
hypertension in pregnancy.  See also table 1 for clinic blood pressure targets for people aged under 80 and table 2 for  clinic blood pressure targets for people aged 80 and over. The tables cover people with  hypertension (with or without type 2 diabetes) as well as people with chronic kidney  dise
--------------------------------------------------------------------------------
--- Keyword Result 2 | ID: [NICE-NG136-2026-CH-041] | Page: 15 ---
1.4.16 Check for postural hypotension (see recommendation 1.1.5) in people with  hypertension and:  • type 2 diabetes or  • symptoms of postural hypotension (see also recommendation 1.1.7) or  • aged 80 and over.  In people with a significant postural drop or symptoms of postural  hypotension, treat
-------------------------------------------------------------------------

### Task 9: Final Retrieval Configuration

Based on the evaluation of multiple queries across different configurations, the optimal retrieval setup for this medical clinical guideline (NICE NG136) is as follows:

* **Chunking Strategy:** A moderate chunk size (e.g., ~1000 characters) with a small overlap (e.g., 100 characters). This preserves the complete context of multi-part medical guidelines and tables without fragmenting crucial clinical targets.
* **Top-K Value:** **k=5**. Our evaluation showed that while $k=3$ works for highly specific questions (P@3 = 0.67 average), complex or deeply nested answers often surface at ranks 4 or 5. Setting $k=5$ ensures high recall without flooding the LLM context window.
* **Retrieval Method:** **Hybrid Search**. The pure dense vector retrieval (`BAAI/bge-small-en-v1.5`) suffered from semantic drift (e.g., retrieving historical committee rationales instead of actual targets for Type 2 Diabetes). Combining dense embeddings with sparse keyword search (BM25) would effectively anchor the semantic search to exact medical terminology and numerical targets.

## Day-3